In [1]:
import numpy as np
import pandas as pd

DATA_DIR = "data"
RNG_SEED = 7
rng = np.random.default_rng(RNG_SEED)

In [2]:
INFLOW_COUNTERPARTY_BY_SECTOR = {
    "dairy": ["cooperative_payment", "customer_payment", "subsidy_payment"],
    "poultry": ["wholesale_buyer_payment", "customer_payment"],
    "food_processing": ["wholesale_buyer_payment", "customer_payment"],
    "handicrafts": ["customer_payment", "wholesale_buyer_payment", "exhibition_sale"],
    "rural_retail": ["customer_payment"],
    "flour_mill": ["customer_payment", "wholesale_buyer_payment"],
    "oil_mill": ["wholesale_buyer_payment", "customer_payment"],
    "textile_weaving": ["wholesale_buyer_payment", "customer_payment"],
    "brick_kiln": ["wholesale_buyer_payment", "contractor_payment"],
    "agri_input_retail": ["customer_payment", "subsidy_payment"],
    "auto_repair": ["customer_payment"],
    "tea_stall_eatery": ["customer_payment"],
}
INFLOW_WEIGHTS = {
    "cooperative_payment": 0.35, "customer_payment": 0.55, "subsidy_payment": 0.05,
    "wholesale_buyer_payment": 0.6, "exhibition_sale": 0.15, "contractor_payment": 0.7,
}

OUTFLOW_TYPES = ["supplier_payment", "utility_bill", "wage_payment", "transport_payment"]
OUTFLOW_WEIGHTS = [0.45, 0.15, 0.30, 0.10]

EMI_CASH_PAYMENT_PROB = 0.08

In [3]:
def _emi_transaction(row) -> dict | None:
    status = row["repayment_status"]
    if status not in ("on_time", "late"):
        return None
    if rng.random() < EMI_CASH_PAYMENT_PROB:
        return None

    emi = float(row["emi_due"])
    period = pd.Period(row["month"], freq="M")
    days_in_month = period.days_in_month

    if status == "on_time":
        amount = emi
        day = int(rng.integers(1, days_in_month + 1))
    else:  # late
        amount = emi * rng.uniform(0.40, 0.85)
        day = int(rng.integers(max(1, days_in_month - 8), days_in_month + 1))

    hour = int(rng.integers(9, 19))
    minute = int(rng.integers(0, 60))
    ts = pd.Timestamp(year=period.year, month=period.month, day=day, hour=hour, minute=minute)
    return {
        "enterprise_id": row["enterprise_id"], "month": row["month"], "timestamp": ts,
        "direction": "outflow", "amount": round(float(amount), 2),
        "counterparty_type": "loan_repayment",
    }

In [4]:
def _split_amount_into_n(total: float, n: int) -> np.ndarray:
    """Split `total` into n positive lognormal-shaped parts summing to `total`."""
    if n <= 0 or total <= 0:
        return np.array([])
    raw = rng.lognormal(mean=0, sigma=0.7, size=n)
    return raw / raw.sum() * total

In [5]:
def _random_timestamps(month_str: str, n: int) -> list[pd.Timestamp]:
    if n <= 0:
        return []
    period = pd.Period(month_str, freq="M")
    days_in_month = period.days_in_month
    # mild bias toward first half of month (income/payments often cluster post-harvest/paydays)
    day_weights = np.array([1.3 if d <= 10 else (1.1 if d <= 20 else 0.8) for d in range(1, days_in_month + 1)])
    day_weights = day_weights / day_weights.sum()
    days = rng.choice(np.arange(1, days_in_month + 1), size=n, p=day_weights)

    # daytime bias: 8am-9pm, peak around midday and early evening
    hour_choices = list(range(8, 21))
    hour_weights = np.array([1.0, 1.2, 1.4, 1.3, 1.1, 1.0, 0.9, 1.0, 1.1, 1.3, 1.4, 1.2, 1.0])
    hour_weights = hour_weights / hour_weights.sum()
    hours = rng.choice(hour_choices, size=n, p=hour_weights)
    minutes = rng.integers(0, 60, size=n)

    return [pd.Timestamp(year=period.year, month=period.month, day=int(d), hour=int(h), minute=int(mi))
            for d, h, mi in zip(days, hours, minutes)]

In [6]:
def generate_transactions_for_row(row, sector: str) -> list[dict]:
    txns = []
    eid = row["enterprise_id"]
    month = row["month"]

    # --- inflows ---
    n_in = int(row["upi_inflow_txn_count"])
    vol_in = float(row["upi_inflow_txn_volume"])
    if n_in > 0 and vol_in > 0:
        amounts = _split_amount_into_n(vol_in, n_in)
        timestamps = _random_timestamps(month, n_in)
        counterparties = INFLOW_COUNTERPARTY_BY_SECTOR.get(sector, ["customer_payment"])
        weights = np.array([INFLOW_WEIGHTS.get(c, 0.3) for c in counterparties])
        weights = weights / weights.sum()
        cp_choices = rng.choice(counterparties, size=n_in, p=weights)

        for amt, ts, cp in zip(amounts, timestamps, cp_choices):
            txns.append({
                "enterprise_id": eid, "month": month, "timestamp": ts,
                "direction": "inflow", "amount": round(float(amt), 2),
                "counterparty_type": cp,
            })

    # --- outflows: digital share of expenses, fewer/lumpier than inflows ---
    digital_expense_fraction = rng.uniform(0.35, 0.65)
    vol_out = float(row["expenses"]) * digital_expense_fraction * (
        float(row["upi_inflow_txn_volume"]) / max(float(row["income"]), 1)  # scale with this enterprise's overall digital activity
    )
    n_out = max(0, int(n_in * rng.uniform(0.25, 0.5))) if n_in > 0 else 0
    if n_out > 0 and vol_out > 0:
        amounts = _split_amount_into_n(vol_out, n_out)
        timestamps = _random_timestamps(month, n_out)
        cp_choices = rng.choice(OUTFLOW_TYPES, size=n_out, p=OUTFLOW_WEIGHTS)
        for amt, ts, cp in zip(amounts, timestamps, cp_choices):
            txns.append({
                "enterprise_id": eid, "month": month, "timestamp": ts,
                "direction": "outflow", "amount": round(float(amt), 2),
                "counterparty_type": cp,
            })

    return txns

In [7]:
def generate_emi_transaction_for_row(row) -> dict | None:
    return _emi_transaction(row)

In [8]:
def main():
    rec = pd.read_csv(f"{DATA_DIR}/monthly_records.csv")
    ent = pd.read_csv(f"{DATA_DIR}/enterprises.csv")[["enterprise_id", "sector"]]
    rec = rec.merge(ent, on="enterprise_id")

    all_txns = []
    for _, row in rec.iterrows():
        all_txns.extend(generate_transactions_for_row(row, row["sector"]))
        emi_txn = generate_emi_transaction_for_row(row)
        if emi_txn is not None:
            all_txns.append(emi_txn)

    txn_df = pd.DataFrame(all_txns)
    txn_df = txn_df.sort_values(["enterprise_id", "timestamp"]).reset_index(drop=True)
    txn_df.insert(0, "transaction_id", ["TXN" + str(i).zfill(8) for i in range(1, len(txn_df) + 1)])

    txn_df.to_csv(f"{DATA_DIR}/transactions.csv", index=False)

    txn_df["month"] = txn_df["month"].astype(str)
    outflows = txn_df[txn_df.direction == "outflow"]

    outflow_agg = outflows.groupby(["enterprise_id", "month"]).agg(
        upi_outflow_txn_count=("transaction_id", "count"),
        upi_outflow_txn_volume=("amount", "sum"),
    ).reset_index()

    loan_repay_agg = outflows[outflows.counterparty_type == "loan_repayment"].groupby(
        ["enterprise_id", "month"]
    )["amount"].sum().rename("loan_repayment_collected_upi").reset_index()

    rec_full = pd.read_csv(f"{DATA_DIR}/monthly_records.csv")
    rec_full = rec_full.merge(outflow_agg, on=["enterprise_id", "month"], how="left")
    rec_full = rec_full.merge(loan_repay_agg, on=["enterprise_id", "month"], how="left")
    rec_full["upi_outflow_txn_count"] = rec_full["upi_outflow_txn_count"].fillna(0).astype(int)
    rec_full["upi_outflow_txn_volume"] = rec_full["upi_outflow_txn_volume"].fillna(0).round(2)
    rec_full["loan_repayment_collected_upi"] = rec_full["loan_repayment_collected_upi"].fillna(0).round(2)
    rec_full.to_csv(f"{DATA_DIR}/monthly_records.csv", index=False)

    print(f"Total transactions: {len(txn_df):,}")
    print(f"Inflows: {(txn_df.direction=='inflow').sum():,}  Outflows: {(txn_df.direction=='outflow').sum():,}")
    print(f"Total inflow volume: {txn_df[txn_df.direction=='inflow']['amount'].sum():,.0f}")
    print(f"Cross-check vs monthly upi_inflow_txn_volume sum: {rec['upi_inflow_txn_volume'].sum():,.0f}")
    print(f"\nSample rows:\n{txn_df.head(8).to_string(index=False)}")


if __name__ == "__main__":
    main()

Total transactions: 151,083
Inflows: 111,749  Outflows: 39,334
Total inflow volume: 103,401,079
Cross-check vs monthly upi_inflow_txn_volume sum: 103,401,078

Sample rows:
transaction_id enterprise_id   month           timestamp direction  amount       counterparty_type
   TXN00000001    GJAH091721 2024-08 2024-08-04 10:46:00   outflow 4751.30          loan_repayment
   TXN00000002    GJAH091721 2024-08 2024-08-21 09:46:00    inflow 1129.95 wholesale_buyer_payment
   TXN00000003    GJAH091721 2024-09 2024-09-12 10:17:00   outflow 4751.30          loan_repayment
   TXN00000004    GJAH091721 2024-09 2024-09-13 10:48:00    inflow  824.87 wholesale_buyer_payment
   TXN00000005    GJAH091721 2024-10 2024-10-01 15:25:00    inflow  991.64 wholesale_buyer_payment
   TXN00000006    GJAH091721 2024-10 2024-10-15 15:27:00   outflow 4751.30          loan_repayment
   TXN00000007    GJAH091721 2024-11 2024-11-11 10:58:00    inflow 1134.58        customer_payment
   TXN00000008    GJAH091721 2024-11